# Global SeapoPym simulation

Run SeapoPym at 1° resolution with the reference parameters of Table 1, on
the temperature and NPP forcings of `data/forcings_global.zarr`.

Inputs: `data/forcings_global.zarr`, `parameters.yaml`.
Output: `data/biomass_global.zarr` (biomass restricted to the analysis period).
Runtime: test mode ~30 s, production mode ~4-5 min.

In [1]:
from datetime import datetime, timedelta
from pathlib import Path

import xarray as xr
import yaml
from dask.distributed import Client
from seapopym.configuration.no_transport import (
    ForcingParameter,
    ForcingUnit,
    FunctionalGroupParameter,
    FunctionalGroupUnit,
    FunctionalTypeParameter,
    KernelParameter,
    MigratoryTypeParameter,
    NoTransportConfiguration,
)
from seapopym.model.no_transport_model import NoTransportModel


def _project_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Project root marker {marker!r} not found.")


PROJECT_ROOT = _project_root()
DATA_DIR = PROJECT_ROOT / "data"

with open(PROJECT_ROOT / "parameters.yaml") as f:
    PARAMS = yaml.safe_load(f)
REF = PARAMS["model_parameters"]["reference"]
GS = PARAMS["global_simulation"]
mode = GS["mode"]
T_START = GS["start_date"]
T_END = GS[mode]["end_date"]
T_ANALYSIS_START = (datetime.strptime(GS["spin_up_end"], "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Mode: {mode} | forcing: {T_START} -> {T_END} | analysis output: {T_ANALYSIS_START} -> {T_END}")

Mode: test | forcing: 1998-01-01 -> 2003-12-31 | analysis output: 2000-01-01 -> 2003-12-31


## Load forcings and configure the model

Temperature is loaded with a singleton `Z` dimension (epipelagic only). The
model applies its internal temperature transformation; no explicit transform
is needed here.

In [2]:
client = Client(n_workers=2, threads_per_worker=1, memory_limit="16GB")
print(client.dashboard_link)

forcings = xr.open_zarr(DATA_DIR / "forcings_global.zarr").sel(T=slice(T_START, T_END)).load()

# SeapoPym expects a (T, Z, Y, X) shape for temperature; Z=0 matches day/night_layer below.
temperature = forcings.temperature.expand_dims(Z=[0], axis=1)
for coord, axis in {"T": "T", "Z": "Z", "Y": "Y", "X": "X"}.items():
    if coord in temperature.coords:
        temperature[coord].attrs["axis"] = axis
    if coord in forcings.npp.coords:
        forcings.npp[coord].attrs["axis"] = axis

functional_group = FunctionalGroupParameter(functional_group=[FunctionalGroupUnit(
    name="zooplankton",
    energy_transfert=REF["energy_transfert"],
    functional_type=FunctionalTypeParameter(
        lambda_temperature_0=REF["lambda_temperature_0"],
        gamma_lambda_temperature=REF["gamma_lambda_temperature"],
        tr_0=REF["tr_0"],
        gamma_tr=REF["gamma_tr"],
    ),
    migratory_type=MigratoryTypeParameter(day_layer=0, night_layer=0),
)])

config = NoTransportConfiguration(
    forcing=ForcingParameter(
        temperature=ForcingUnit(forcing=temperature),
        primary_production=ForcingUnit(forcing=forcings.npp),
    ),
    functional_group=functional_group,
    kernel=KernelParameter(compute_initial_conditions=True),
)

http://127.0.0.1:8787/status


npp unit is milligram / day / meter ** 2, it will be converted to gram / day / meter ** 2.


npp unit is milligram / day / meter ** 2, it will be converted to gram / day / meter ** 2.


## Run and persist the biomass (analysis period only)

In [3]:
with NoTransportModel.from_configuration(configuration=config) as model:
    model.run()
    model.state.compute()
    biomass = model.state.biomass.load().copy()

output_path = DATA_DIR / "biomass_global.zarr"
biomass.sel(T=slice(T_ANALYSIS_START, T_END)).to_zarr(output_path, mode="w", zarr_format=2)
client.close()
print(f"Wrote {output_path.name} | size = {sum(f.stat().st_size for f in output_path.rglob('*') if f.is_file()) / 1e6:.1f} MB")

Wrote biomass_global.zarr | size = 464.7 MB
